In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import numpy as np
import mesh, fd_validation
import closest_point_projection

In [ ]:
K = 3 # Manifold dimension
D = 2 # Embedding dimension for K = 2 case

# Validate closest point projections

In [ ]:
# Construct the "target" manifold
if K == 3:
    manifold = mesh.Mesh('../../misc/examples/meshes/lilium.msh')
    proj = closest_point_projection.ClosestPointProjection(manifold)
    D = 3
else:
    nv = 50
    t = np.linspace(0, np.pi, nv)
    V = np.column_stack([np.sqrt(t) * np.cos(2 * t), np.sqrt(t) * np.sin(2 * t), t - np.pi])[:, :D]
    E = np.column_stack((np.arange(nv - 1), np.arange(nv - 1) + 1))
    manifold = (V, E)
    proj = closest_point_projection.ClosestPointProjection(*manifold)

In [ ]:
Q = np.random.normal(size=(40, D))
P = np.array([proj.project(q).p for q in Q])

In [ ]:
import viewer

v = viewer.Viewer(manifold, wireframe=True)
sv = viewer.LineMeshViewer((np.vstack((Q, P)), [(qi, qi + len(Q)) for qi in range(len(Q))]), superView=v)
sv.showPoints()

v.show()

In [ ]:
r = proj.project(Q[2, :])
print(r.barycoords)
print(np.linalg.eigh(r.dp_dq))

# Construct and validate closest-point `ProjectedSprings`

In [ ]:
import loads
import elastic_sheet, elastic_solid
import energy

In [ ]:
if D == 3: query_es = elastic_sheet.ElasticSheet(mesh.Mesh('../../misc/examples/meshes/bunny_coarse.msh').boundaryMesh(), energy.NeoHookeanYoungPoisson(2, 0, 0))
if D == 2: query_es = elastic_solid.ElasticSolid(mesh.Mesh('../../misc/examples/meshes/square_hole.off'),                 energy.NeoHookeanYoungPoisson(2, 0, 0))

In [ ]:
dsm = query_es.deformationSamplerMatrix(Q)
s = loads.ProjectedSprings(query_es, dsm, proj)

In [ ]:
prob = query_es.EquilibriumProblem([s])

In [ ]:
prob.energy()

In [ ]:
import fd_validation
fd_validation.gradConvergencePlot(prob)

In [ ]:
fd_validation.hessConvergencePlot(prob)